# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset describes protein abundance, modification, and sequence characteristics derived from human extracellular vesicle samples.

### Dataset Source

The dataset source is provided via a Croissant schema URL.

**Schema URL:**

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset high-level information
print('Dataset Title:', dataset.metadata.name)
print('Dataset Description:', dataset.metadata.description)
print('Published:', getattr(dataset.metadata, 'datePublished', 'N/A'))

## 2. Data Overview

Review available record sets, their fields, and respective `@id`s for reference.

This section prints all available record sets, listing their `@id` and the fields within each, also referenced by `@id` (as required for reproducibility and referencing downstream).

In [ ]:
print("Available record sets and fields in the dataset:")
record_sets = list(dataset.record_sets)
all_record_set_ids = []
for rs in record_sets:
    rs_id = rs.id
    all_record_set_ids.append(rs_id)
    print(f"  RecordSet @id: {rs_id}")
    print(f"    Title: {getattr(rs, 'name', '')}")
    if hasattr(rs, 'fields'):
        for f in rs.fields:
            print(f"      Field @id: {f.id}")
            print(f"        - name: {f.name}")
            print(f"        - dataType: {getattr(f, 'data_type', '')}")
    print("")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis using the record set's and fields' `@id` values. This enables you to process each table of the dataset in a reproducible way.

> **Note:** Use the `@id` values output above to select which record sets you want to analyze. For demonstration, we'll extract all available data tables.

In [ ]:
# Get all record set @ids
from pprint import pprint

dataframes = {}

if not all_record_set_ids:
    # If the previous cell did not execute, regenerate
    all_record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in all_record_set_ids:
    df = pd.DataFrame(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = df
    print(f"--- Loaded DataFrame for RecordSet @id: {record_set_id} ---")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())  # Show sample data

## 4. Exploratory Data Analysis (EDA)

Apply data processing to one of the record sets. For demonstration, we use the first record set and choose a numeric field for common operations such as filtering, normalization, and grouping. Please update the variables to match actual `@id`s as shown in the earlier outputs.

In [ ]:
# Example: Use the first record set and find a numeric field
record_set_id = all_record_set_ids[0]
df = dataframes[record_set_id]

print(f"Analyzing RecordSet @id: {record_set_id}")
print("DataFrame columns and types:")
print(df.dtypes)

# Attempt to find a numeric field (float or int)
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
if not numeric_columns:
    print("No numeric columns found!")
else:
    numeric_field = numeric_columns[0]
    print(f"Using numeric field: {numeric_field}")

    # Example threshold filter: greater than the mean
    threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records where {numeric_field} > mean ({threshold}):")
    display(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping if another suitable column exists
    group_columns = [c for c in df.columns if c != numeric_field]
    if group_columns:
        group_field = group_columns[0]
        print(f"\nGrouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        display(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field or visualize the relationship between two fields using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field' in locals() and numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # If group_field exists, show boxplot
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to:
- Load a Croissant dataset using the `mlcroissant` library
- Explore the available record sets and their fields using each entity's `@id`
- Extract tabular data and inspect data types
- Perform common EDA operations such as filtering, normalization, and grouping
- Visualize results

For deeper analysis, consult the specific `@id` values and documentation of each field in the Croissant metadata, and adapt the analysis/visualization as required for your scientific objectives.